# Dog Registrations, Names, Demographic and Breed Information from Zurich City, 2014–2024 Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset citation:** Wielgosz, P. 2026 Dog Registrations, Names, Demographic and Breed Information from Zurich City, 2014–2024. Frontiers.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.s5da-e6vr/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata summary
meta = dataset.metadata
print(f"{meta.name} (Version: {getattr(meta, 'version', 'N/A')})")
print(f"Published: {getattr(meta, 'datePublished', 'N/A')}")
print(f"Description: {meta.description}\n")
print(f"Record sets (@id): {[r['@id'] for r in getattr(meta, 'record_sets', [])]}\n")
print(f"Available distribution resources (@id): {[d['@id'] for d in getattr(meta, 'distribution', [])]}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Here, we print the list of record sets, and for each, its fields and columns by `@id`.


In [ ]:
# Retrieve all record set @ids
record_sets = [rs['@id'] for rs in getattr(meta, 'record_sets', [])]

if not record_sets:
    # If metadata.record_sets is empty, let's infer from the schema
    # We'll look for record sets inside dataset.metadata._jsonld
    schema = dataset.metadata._jsonld
    # Most Croissant datasets have 'recordSet' field in the top-level @graph
    # Find all top-level objects of type cr:RecordSet
    record_sets_explicit = []
    graph = schema.get('@graph', [])
    for entry in graph:
        if entry.get('@type') in ['cr:RecordSet', 'RecordSet']:
            record_sets_explicit.append(entry['@id'])
    record_sets = record_sets_explicit
    print(f"Found record_set @ids: {record_sets}\n")
else:
    print(f"Found record_set @ids: {record_sets}\n")

# For each record set, show its field @ids
for rs_id in record_sets:
    print(f"Record Set: {rs_id}")
    
    # Find the record set object in graph
    rs_obj = None
    for entry in graph:
        if entry.get('@id') == rs_id:
            rs_obj = entry
            break
    if rs_obj:
        # Fields
        field_ids = rs_obj.get('field', [])
        if isinstance(field_ids, dict):
            field_ids = [field_ids]
        if field_ids:
            print("  Fields (by @id):")
            for f in field_ids:
                if isinstance(f, dict):
                    print(f"    - {f.get('@id', f)}")
                else:
                    print(f"    - {f}")
        else:
            print("  No fields listed.")
        # Columns (from fields)
        col_ids = []
        for entry in graph:
            if entry.get('@type') in ['cr:Field', 'Field'] and entry.get('isPartOf') == rs_id:
                # Each field may refer to a column
                col_id = entry.get('column', None)
                if isinstance(col_id, list):
                    col_ids.extend(col_id)
                elif col_id:
                    col_ids.append(col_id)
        if col_ids:
            print("  Columns (by @id):")
            for c in col_ids:
                if isinstance(c, dict):
                    print(f"    - {c.get('@id', c)}")
                else:
                    print(f"    - {c}")
    else:
        print("  Record set details not found in JSON-LD graph.")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Let's show a sample from each record set. If there are more than one, you can extract each into a pandas DataFrame.


In [ ]:
dataframes = {}
loaded_record_sets = []
for rs_id in record_sets:
    print(f"Loading records for record set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            loaded_record_sets.append(rs_id)
            print(f"  Loaded {len(df)} records. Sample columns: {df.columns.to_list()[:6]}\n")
            display(df.head())
        else:
            print("  No records found for this record set.\n")
    except Exception as e:
        print(f"  Could not load records: {e}\n")
if loaded_record_sets:
    print(f"Loaded DataFrames for record_sets: {loaded_record_sets}")
else:
    print("No DataFrames loaded. Check if the dataset contains accessible records and record_set IDs are correct.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We'll pick the first record set with accessible data and explore numeric and categorical attributes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Select first record set with data for EDA
if loaded_record_sets:
    record_set_id = loaded_record_sets[0]
    df = dataframes[record_set_id]
    print(f"EDA for record_set: {record_set_id}, {df.shape[0]} records.\n")
    # Show list of integer and float columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric columns in the DataFrame: {numeric_cols}")
    # Select numeric field for demonstration (try typical, else first numeric column)
    candidate_fields = ['Anzahl', 'HundAnzahl', 'count', 'AlterVHundNum', 'Jahrhund', 'Geburtsjahr', 'birth_year']
    numeric_field = None
    for col in candidate_fields:
        if col in df.columns:
            numeric_field = col
            break
    if not numeric_field and numeric_cols:
        numeric_field = numeric_cols[0]
    print(f"Selected numeric field for demonstration: {numeric_field}")
    # Filter (e.g., records with value > threshold)
    if numeric_field:
        threshold = df[numeric_field].quantile(0.75) if df[numeric_field].nunique() > 10 else df[numeric_field].max()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold}")
        display(filtered_df.head())
        # Normalization
        df[f"{numeric_field}_normalized"] = (df[numeric_field] - df[numeric_field].mean()) / df[numeric_field].std()
        print(f"Mean and std of {numeric_field}: {df[numeric_field].mean():.2f}, {df[numeric_field].std():.2f}")
        print(df[[numeric_field, f"{numeric_field}_normalized"]].head())
    else:
        print("No numeric field found for normalization.")
    # Select categorical grouping field (e.g., 'Geschlecht', 'Sex', 'GeschlechtText', 'RasseText')
    candidate_groups = ['Geschlecht', 'Sex', 'GeschlechtText', 'RasseText', 'breed', 'district', 'Stadtkreis', 'Quartier']
    group_field = None
    for g in candidate_groups:
        if g in df.columns:
            group_field = g
            break
    print(f"Selected group field for demonstration: {group_field}")
    # Grouped mean
    if group_field and numeric_field:
        grouped_df = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        print(f"Grouped mean of {numeric_field} by {group_field}:\n{grouped_df.head(10)}\n")
else:
    print("No loaded DataFrame found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot the distribution of a numeric field and the top categories by count.


In [ ]:
if loaded_record_sets:
    # Histogram of a numeric variable
    if numeric_field:
        plt.figure(figsize=(7,4))
        df[numeric_field].hist(bins=30, color='skyblue', edgecolor='black')
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
    # Barplot for a categorical field
    if group_field:
        top_cats = df[group_field].value_counts().head(10)
        top_cats.plot(kind='bar', figsize=(8,4), color='coral', edgecolor='black')
        plt.title(f"Top 10 values of group field: {group_field}")
        plt.ylabel("Count")
        plt.show()

## 6. Conclusion
In this exploratory notebook, we leveraged the `mlcroissant` library to programmatically load and analyze the [Dog Registrations, Names, Demographic and Breed Information from Zurich City, 2014–2024](https://sen.science/doi/10.71728/senscience.s5da-e6vr/fair2.json) dataset. We programmatically accessed metadata, inspected structure, extracted tabular data, performed basic filtering, normalization, and grouped analysis, and visualized key attributes such as numeric field distributions and categorical frequencies.

You can use and extend this notebook for more detailed analyses, advanced visualizations, or to prepare the data for downstream modeling tasks. Refer to the dataset's Croissant schema and the [mlcroissant documentation](https://mlcommons.github.io/croissant/) for advanced usage.
